#Llama2 Model


#1.Imports and loads

In [9]:
#!pip install -q transformers==4.41.2 accelerate==0.30.0 datasets==2.19.1 peft==0.11.1 bitsandbytes==0.43.1 trl==0.8.6


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.4/302.4 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 100.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.


In [2]:
from huggingface_hub import login
login()

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
from datasets import load_dataset

#2. Dataset load

In [2]:
from datasets import load_dataset

ds = load_dataset("utcn-chatbot/llama2-dataset")  # repo privat pe hugging faceul meu (se poate face load si din github)
print(ds)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/27.0 [00:00<?, ?B/s]

EmptyDatasetError: The directory at hf://datasets/utcn-chatbot/llama2-dataset@3abaaefd5b9f34ab9d3baba5b7047e50b26484d3 doesn't contain any data files

#3.Load base model

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

BASE = "meta-llama/Llama-2-7b-chat-hf"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tok = AutoTokenizer.from_pretrained(BASE, use_auth_token=True)
tok.pad_token = tok.eos_token

base = AutoModelForCausalLM.from_pretrained(
    BASE,
    device_map="auto",
    quantization_config=bnb,
    torch_dtype=torch.bfloat16,
    use_auth_token=True,
)



/usr/local/lib/python3.12/dist-packages/transformers/models/auto/tokenization_auto.py:1001: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

#4.LoRA Adapters
-We attach LoRA adapters to the model these are the only weights we’ll train.

In [5]:
from peft import LoraConfig, get_peft_model

lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    task_type="CAUSAL_LM"
)
model = get_peft_model(base, lora_cfg)


#5. Format dataset into Prompts
-We convert each JSONL entry into a chat-style prompt so the model learns to answer as UTCN assistant.

In [6]:
SYS = "Ești asistentul oficial UTCN. Răspunde concis și corect."

def format_example(ex):
    inst = ex["instruction"].strip()
    inp  = (ex.get("input") or "").strip()
    out  = ex["output"].strip()
    if inp:
        prompt = f"<<SYS>>{SYS}<</SYS>>\nUser: {inst}\n{inp}\nAssistant:"
    else:
        prompt = f"<<SYS>>{SYS}<</SYS>>\nUser: {inst}\nAssistant:"
    return prompt + " " + out


#6. Data Collator
-We want the model to only learn the assistant’s answer, not the question.

In [7]:
from trl import DataCollatorForCompletionOnlyLM

collator = DataCollatorForCompletionOnlyLM(
    response_template="Assistant:",
    tokenizer=tok
)


#7. Training Arguments

In [1]:
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir="out-lora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=2,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=20,
    save_steps=200,
    evaluation_strategy="steps",
    eval_steps=200,
    bf16=True,
    gradient_checkpointing=True,
)


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


#8.Train with SFTTrainer
-We use Hugging Face TRL’s SFTTrainer (supervised fine-tuning).

In [2]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tok,
    train_dataset=ds["train"],
    eval_dataset=ds.get("validation") or ds.get("eval"),
    formatting_func=lambda batch: [format_example(x) for x in (batch if isinstance(batch, list) else [batch])],
    max_seq_length=1024,
    packing=True,
    data_collator=collator,
    args=args
)

trainer.train()


NameError: name 'model' is not defined

#9.Save Adapters
-We save only the LoRA adapters, not the full LLaMA-2 model.

In [3]:
trainer.model.save_pretrained("out-lora")
tok.save_pretrained("out-lora")


NameError: name 'trainer' is not defined

#10. Push to Hugging Face
-I have a private repo on hugging face with adapters and the dataset.

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
api.upload_folder(
    folder_path="out-lora",
    repo_id="utcn-chatbot/llama2-chatbot",
    repo_type="model"
)
